In [32]:
import pandas as pd
from scipy.stats import chi2_contingency,pointbiserialr
import numpy as np
df = pd.read_csv("../data/raw/bank-full.csv",sep=";")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 8.1 MB


In [33]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [34]:
print(f"Number of customers who subscribed to the service is {len(df[df['y']=='yes'])} and the class percentage is {len(df[df['y']=='yes'])/len(df)*100:.2f}%")
print(f"Number of customers who did not subscribe to the service: {len(df[df['y']=='no'])} and the class percentage is {len(df[df['y']=='no'])/len(df)*100:.2f}%")

Number of customers who subscribed to the service is 5289 and the class percentage is 11.70%
Number of customers who did not subscribe to the service: 39922 and the class percentage is 88.30%


In [35]:
txt_cols = df.select_dtypes(include=['str']).columns
num_cols = df.select_dtypes(include=['int', 'float']).columns

In [36]:
for col in txt_cols:
    print(f"Column {col} has {df[col].nunique()} unique values : {list(df[col].unique())}")

Column job has 12 unique values : ['management', 'technician', 'entrepreneur', 'blue-collar', 'unknown', 'retired', 'admin.', 'services', 'self-employed', 'unemployed', 'housemaid', 'student']
Column marital has 3 unique values : ['married', 'single', 'divorced']
Column education has 4 unique values : ['tertiary', 'secondary', 'unknown', 'primary']
Column default has 2 unique values : ['no', 'yes']
Column housing has 2 unique values : ['yes', 'no']
Column loan has 2 unique values : ['no', 'yes']
Column contact has 3 unique values : ['unknown', 'cellular', 'telephone']
Column month has 12 unique values : ['may', 'jun', 'jul', 'aug', 'oct', 'nov', 'dec', 'jan', 'feb', 'mar', 'apr', 'sep']
Column poutcome has 4 unique values : ['unknown', 'failure', 'other', 'success']
Column y has 2 unique values : ['no', 'yes']


In [37]:
for col in txt_cols:
    df[col] = df[col].astype('category').cat.codes

In [38]:
cat_results = []

for col in txt_cols:
    table = pd.crosstab(df[col], df["y"])

    chi2, p_value, dof, expected = chi2_contingency(table)

    n = table.sum().sum()
    r, k = table.shape

    # Cramér's V
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))

    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)

    cramers_v = np.sqrt(
        phi2corr / min(kcorr - 1, rcorr - 1)
    )

    cat_results.append({
        "feature": col,
        "chi2": chi2,
        "p_value": p_value,
        "cramers_v": cramers_v
    })

cat_results = (
    pd.DataFrame(cat_results)
    .sort_values("cramers_v", ascending=False)
)

display(cat_results)

,feature,chi2,p_value,cramers_v
9,y,45201.319916,0.000000e+00,0.999893
8,poutcome,4391.506589,0.000000e+00,0.311560
7,month,3061.838938,0.000000e+00,0.259772
6,contact,1035.714225,1.251738e-225,0.151211
4,housing,874.822449,2.918798e-192,0.139025
0,job,836.105488,3.337122e-172,0.135094
2,education,238.923506,1.626656e-51,0.072238
5,loan,209.616980,1.665061e-47,0.067929
1,marital,196.495946,2.145100e-43,0.065590
3,default,22.202250,2.453861e-06,0.021656


In [39]:
df[num_cols].describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


In [40]:
#outlayer checking
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"Column {col} has {len(outliers)} outliers")

Column age has 487 outliers
Column balance has 4729 outliers
Column day has 0 outliers
Column duration has 3235 outliers
Column campaign has 3064 outliers
Column pdays has 8257 outliers
Column previous has 8257 outliers


In [41]:
for col in num_cols:
    corr,pvalue = pointbiserialr(df[col], df['y'])
    print(f"Column {col}: correlation = {corr:.3f}, p-value = {pvalue:.3f}")

Column age: correlation = 0.025, p-value = 0.000
Column balance: correlation = 0.053, p-value = 0.000
Column day: correlation = -0.028, p-value = 0.000
Column duration: correlation = 0.395, p-value = 0.000
Column campaign: correlation = -0.073, p-value = 0.000
Column pdays: correlation = 0.104, p-value = 0.000
Column previous: correlation = 0.093, p-value = 0.000
